# Старый вариант самописной нейронки на Numpy

In [ ]:
class Linear_11(Layer):
    """Полносвязный (линейный) слой: y = x @ W.T + b.
       Основная логика - применение аффинного линейного пространства к входящим данным: y=xA^T+b
    """
    def __init__(
        self, 
        in_features: int,
        out_features: int,
        bias: bool = True
    ) -> None:
        """
        [RU] Инициализация весов для NumPy-based neural network с Xavier initialization для ReLU.
        [EN] Initializes the weights for NumPy-based neural network.
        
        Args:
            in_features: Размер входного тензора
            out_features: Размер выходного тензора
            bias: использовать ли смещение (по умолчанию True)
        """
        self.in_features: int = in_features
        self.out_features: int = out_features
        self.use_bias: bool = bias

        # Xavier uniform initialization
        # В PyTorch используется k = 1/in_features для равномерного распределения
        k: float = 1.0 / in_features
        limit: float = np.sqrt(k)
        self.weights: npt.NDArray[np.float64] = np.random.uniform(
            -limit, 
            limit, 
            (out_features, in_features)
        )

        # Инициализация смещений нулями
        if bias:
            self.bias: Optional[npt.NDArray[np.float64]] = np.random.uniform(
                -limit, 
                limit, 
                (out_features,)
            )
        else:
            self.bias = None
        
        # Для хранения входных данных для backward
        self.input: Optional[npt.NDArray[np.float64]] = None
        self.grad_weights: Optional[npt.NDArray[np.float64]] = None
        self.grad_bias: Optional[npt.NDArray[np.float64]] = None

    def forward(
        self,
        x: npt.NDArray[np.float64]
    ) -> npt.NDArray[np.float64]:
        """
        Прямой проход: y = x @ W.T + b.
        
        Args:
            x: Входной тензор [batch_size, in_features]
        
        Returns:
            Выходной тензор [batch_size, out_features]
        """
        self.input = x.copy()

        # Вычисляем x @ W.T
        output = np.dot(x, self.weights.T)

        if self.bias is not None:
            output = output + self.bias
        return output

    def backward(
        self,
        grad_output: npt.NDArray[np.float64]
    ) -> Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64], npt.NDArray[np.float64]]:
        """
        Обратный проход: вычисление градиентов.
        
        Args:
            grad_output: Градиент с следующего слоя [batch_size, out_features]
        
        Returns:
            Tuple[grad_weights, grad_bias, grad_input]
        """
        if self.input is None:
            raise RuntimeError("Forward pass must be called before backward")
        
        # Градиент по весам: input.T @ grad_output
        if self.grad_weights is None:
            self.grad_weights = np.dot(grad_output.T, self.input)
        else:
            self.grad_weights += np.dot(grad_output.T, self.input)

        # Градиент по смещению: sum(grad_output, axis=0)
        if self.bias is not None:
            grad_bias = np.sum(grad_output, axis=0)
            if self.grad_bias is None:
                self.grad_bias = grad_bias
            else:
                self.grad_bias += grad_bias

        # Градиент по входу: grad_output @ weights
        grad_input: npt.NDArray[np.float64] = np.dot(grad_output, self.weights)
        return grad_input
    
    def get_params(self) -> List[npt.NDArray[np.float64]]:
        """Возвращает параметры (веса и смещения)."""
        params = [self.weights]
        if self.bias is not None:
            params.append(self.bias)
        return params
    
    def set_params(
        self, 
        gradients: List[npt.NDArray[np.float64]]
    ) -> None:
        """Устанавливает градиенты параметров."""
        self.grad_weights, self.grad_bias = gradients
    
    def get_gradients(self) -> Tuple[npt.NDArray[np.float64], npt.NDArray[np.float64]]:
        """Возвращает сохраненные градиенты."""
        grads = [self.grad_weights]
        if self.bias is not None:
            grads.append(self.grad_bias)
        return grads
    
    def zero_grad(self) -> None:
        """Обнуляет сохраненные градиенты."""
        self.grad_weights = None
        self.grad_bias = None
        self.input = None

    def __repr__(self) -> str:
        """Строковое представление как в PyTorch."""
        bias_str = "True" if self.bias is not None else "False"
        return f"Linear(in_features={self.in_features}, out_features={self.out_features}, bias={bias_str})"

In [ ]:
class NumpyNeuralNetwork_1:
    """
    Полносвязная нейронная сеть на NumPy.
    Архитектура: Linear(784,512) -> ReLU -> Linear(512,512) -> ReLU -> Linear(512,10)
    """
    def __init__(
        self,
        input_size: int = MY_CONSTANTS.input_size,
        hidden_size: int = MY_CONSTANTS.hidden_size,
        output_size: int = MY_CONSTANTS.output_size,
        learning_rate: float = MY_CONSTANTS.learning_rate
    ) -> None:
        """
        Инициализация нейронной сети.
        
        Args:
            input_size: Размер входного слоя (784 для FashionMNIST)
            hidden_size: Размер скрытых слоев (512)
            output_size: Размер выходного слоя (10 классов)
            learning_rate: Скорость обучения
        """
        self.learning_rate: float = learning_rate
        self.input_size: int = input_size
        self.hidden_size: int = hidden_size
        self.output_size: int = output_size
        
        self.layers: List[Union[Linear_11, ReLU]] = [
            Linear_11(input_size, hidden_size), # Первый линейный слой
            ReLU(), # Первое использование ReLU
            Linear_11(hidden_size, hidden_size),  # Второй линейный слой
            ReLU(), # Второе использование ReLU
            Linear_11(hidden_size, output_size) # Выходной слой
        ]

        self.linear_layers: List[Linear_11] = [
            layer for layer in self.layers 
            if isinstance(layer, (Linear_11))
        ]

        # Для хранения метрик
        self.train_losses: List[float] = []
        self.test_losses: List[float] = []
        self.train_accuracies: List[float] = []
        self.test_accuracies: List[float] = []
        self.train_mses: List[float] = []
        self.test_mses: List[float] = []
        self.train_precisions: List[float] = []
        self.test_precisions: List[float] = []
        self.train_recalls: List[float] = []
        self.test_recalls: List[float] = []
        self.train_f1s: List[float] = []
        self.test_f1s: List[float] = []
        self.epochs_completed: List[int] = []
        self.batch_losses: List[float] = []
        self.batch_indices: List[int] = []

    """
    [EN] Softmax function for output layer.
    softmax(x)i = exp(x_i - max(x)) / sum{j=1}^{K} exp(x_j - max(x))
    [RU] Использование функции SoftMax
    """
    @staticmethod
    def softmax(
        x: npt.NDArray[np.float64]
    ) -> npt.NDArray[np.float64]:
        """
        Softmax функция с численной стабилизацией.
        softmax(x)i = exp(x_i - max(x)) / sum{j=1}^{K} exp(x_j - max(x)).
        
        Args:
            x: Входной тензор [batch_size, num_classes]
        
        Returns:
            Вероятности классов [batch_size, num_classes]
        """
        # Вычитаем максимум для численной стабильности
        shifted: npt.NDArray[np.float64] = x - np.max(x, axis=1, keepdims=True)
        exp_x: npt.NDArray[np.float64] = np.exp(shifted)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    """
    [EN] Cross-entropy loss function.
    Using categorical cross entropy function:
    -1/N * sum({j=1}^{N} ; log(predicted[y_i]))
    [RU] Функция cross-энтропии для потерь
    """
    @staticmethod
    def cross_entropy_loss(
        predicted: npt.NDArray[np.float64],
        target: npt.NDArray[np.int64]
    )-> float:
        """
        Функция потерь Cross-Entropy.
        
        Args:
            predicted: Предсказанные вероятности [batch_size, num_classes]
            target: Истинные метки [batch_size]
        
        Returns:
            Значение loss (скаляр)
        """
        # Клиппинг для численной стабильности
        predicted = np.clip(predicted, MY_CONSTANTS.tolerance, 1 - MY_CONSTANTS.tolerance)
        batch_size: int = target.shape[0]
        log_likelihood: npt.NDArray[np.float64] = -np.log(
            predicted[range(batch_size), target]
        )
        return float(np.sum(log_likelihood) / batch_size)

    """
    [EN] Forward pass through the network.
    [RU] Прямой ход созданной нейронной сети
    """
    def forward(
        self, 
        input_data: npt.NDArray[np.float64]
    ) -> Tuple[npt.NDArray[np.float64], List[npt.NDArray[np.float64]]]:
        """
        Прямой проход через всю сеть.
        
        Args:
            input_data: Входные данные [batch_size, input_size]
        
        Returns:
            Tuple[output, activations]
            output: Выход сети после softmax [batch_size, output_size]
            activations: Все промежуточные активации
        """
        x: npt.NDArray[np.float64] = input_data
        activations: List[npt.NDArray[np.float64]] = [x]

        # Проход через все слои
        for layer in self.layers:
            x = layer.forward(x)
            activations.append(x)

        output: npt.NDArray[np.float64] = self.softmax(x)
        return output, activations

    """
    [EN] Backward pass for backpropagation.
    [RU] Обратный ход созданной нейронной сетим
    """
    def backward(
        self,
        target_onehot: npt.NDArray[np.float64],
        output: npt.NDArray[np.float64],
    ) -> List[Dict[str, npt.NDArray[np.float64]]]:
        """
        Обратный проход (backpropagation).
        
        Args:
            input_data: Входные данные
            target_onehot: One-hot истинные метки
            output: Выход сети
            activations: Все активации с forward
        
        Returns:
            Список градиентов для каждого линейного слоя
        """
        batch_size: int = output.shape[0]
        gradients: List[Dict[str, npt.NDArray[np.float64]]] = []

        # Градиент для Cross-Entropy + Softmax
        # Начальный градиент от ошибки
        # Вычисляет начальную ошибку, которая затем используется для корректировки весов сети и улучшения ее способности делать точные прогнозы. Этот пример подходит для MSE и для перекрестной энтропии с сигмоидой, когда выход находится между 0 и 1.
        # Квадратичная ошибка (Mean Squared Error - MSE): Loss = 0.5 * (output - target)^2  output_error = output - target
        # Перекрестная энтропия (Cross-Entropy Loss): Используется для задач классификации. Формула немного сложнее, но для сигмоидной функции активации в выходном слое: output_error = output - target
        grad: npt.NDArray[np.float64] = (output - target_onehot) / batch_size

        # Обратное распространение через слои в обратном порядке
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    """
    [EN] Applies the calculated gradients to update weights and biases.
    [RU] Обновление вычисленных градиентов для обновления весов и сдвигов
    """
    def apply_gradients(
        self,
    ) -> None:
        """
        Обновление весов с помощью SGD.
        
        Args:
            gradients: Список градиентов для каждого линейного слоя
        """
        for layer in self.linear_layers:
            grad_weights, grad_bias = layer.get_gradients()
            if grad_weights is not None and grad_bias is not None:
                layer.weights -= self.learning_rate * grad_weights
                layer.bias -= self.learning_rate * grad_bias
                layer.zero_grad()

    def train_epoch(
        self,
        train_images: npt.NDArray[np.float32],
        train_labels: npt.NDArray[np.int64],
        batch_size: int,
        report_interval: int = 100,
        epoch_num: int = 1,
        device: torch.device = torch.device('cuda')
    ) -> Tuple[float, float, List[float], List[int], List[float], float, float, float, float, float, float, float]:
        """
        Обучение на одной эпохе с полным расчетом всех метрик.
        
        Args:
            train_images: Тренировочные изображения
            train_labels: Тренировочные метки
            batch_size: Размер батча
            report_interval: Частота вывода отчета
            epoch_num: Номер текущей эпохи
        
        Returns:
            Tuple[avg_loss, accuracy, batch_losses, batch_indices, mse_values, 
                mean_loss, median_loss, mean_output_loss, median_output_loss,
                avg_mse, precision, recall, f1]
        """
        precision_metric = Precision(task="multiclass", num_classes=10, average='macro').to(device)
        recall_metric = Recall(task="multiclass", num_classes=10, average='macro').to(device)
        f1_metric = F1Score(task="multiclass", num_classes=10, average='macro').to(device)

        num_samples: int = train_images.shape[0]
        num_batches: int = (num_samples + batch_size - 1) // batch_size
        
        # Перемешиваем данные
        permutation: npt.NDArray[np.int64] = np.random.permutation(num_samples)
        images_shuffled: npt.NDArray[np.float32] = train_images[permutation]
        labels_shuffled: npt.NDArray[np.int64] = train_labels[permutation]
        
        total_loss: float = 0.0
        correct_predictions: int = 0
        batch_losses: List[float] = []
        batch_indices: List[int] = []
        output_batch_losses: List[float] = []
        all_predictions: List[int] = []
        all_labels: List[int] = []
        total_mse: float = 0.0
        mse_batch_values: List[float] = []
        
        start_epoch_time: float = time.time()
        
        for batch_idx, i in enumerate(range(0, num_samples, batch_size)):
            batch_X: npt.NDArray[np.float32] = images_shuffled[i:i + batch_size]
            batch_y: npt.NDArray[np.int64] = labels_shuffled[i:i + batch_size]
            
            # Конвертируем в float64 для вычислений
            batch_X_f64: npt.NDArray[np.float64] = batch_X.astype(np.float64)
            
            # One-hot encoding меток
            batch_y_onehot: npt.NDArray[np.float64] = label_encode(batch_y).astype(np.float64)
            
            # Прямой проход
            output, activations = self.forward(batch_X_f64)
            
            # Вычисление loss
            loss: float = self.cross_entropy_loss(output, batch_y)
            total_loss += loss
            batch_losses.append(loss)
            batch_indices.append(batch_idx + 1)
            
            # Точность
            predictions: npt.NDArray[np.int64] = np.argmax(output, axis=1)
            correct_predictions += np.sum(predictions == batch_y)

            # Сохраняем для метрик
            all_predictions.extend(predictions.tolist())
            all_labels.extend(batch_y.tolist())

            # Конвертируем numpy в torch тензоры
            pred_tensor = torch.from_numpy(predictions).to(device)
            target_tensor = torch.from_numpy(batch_y).to(device)
            
            # Обновляем метрики
            precision_metric.update(pred_tensor, target_tensor)
            recall_metric.update(pred_tensor, target_tensor)
            f1_metric.update(pred_tensor, target_tensor)

            # Расчёт MSE для батча
            mse: float = float(np.mean(np.square(output - batch_y_onehot)))
            total_mse += mse
            mse_batch_values.append(mse)
            
            # Обратный проход и обновление весов
            self.backward(batch_y_onehot, output)
            self.apply_gradients()
            
            # Отчет
            if batch_idx % report_interval == 0:
                current: int = min(i + batch_size, num_samples)
                print(f"Epoch {epoch_num} - Loss: {loss:.7f}  [{current:>5d}/{num_samples:>5d}]")
                output_batch_losses.append(loss)

        precision = precision_metric.compute().item()
        recall = recall_metric.compute().item()
        f1 = f1_metric.compute().item()
        
        # Сброс метрик для следующей эпохи
        precision_metric.reset()
        recall_metric.reset()
        f1_metric.reset()
        
        avg_loss: float = total_loss / num_batches
        accuracy: float = correct_predictions / num_samples
        
        # Статистики по loss
        mean_loss: float = float(np.mean(batch_losses))
        median_loss: float = float(np.median(batch_losses))
        mean_output_loss: float = float(np.mean(output_batch_losses)) if output_batch_losses else 0.0
        median_output_loss: float = float(np.median(output_batch_losses)) if output_batch_losses else 0.0
        
        # Средняя MSE по эпохе
        avg_mse: float = total_mse / num_batches
        
        # Precision, Recall, F1-Score
        # precision: float = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
        # recall: float = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
        # f1: float = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
        
        # Время выполнения эпохи
        end_epoch_time: float = time.time()
        total_epoch_time: float = end_epoch_time - start_epoch_time
        minutes: int = int(total_epoch_time // 60)
        seconds: float = total_epoch_time % 60
        
        # Вывод расширенной статистики
        print(f"\n📊 Training Epoch {epoch_num} Statistics:")
        print(f"   • Loss - Mean: {mean_loss:.7f}, Median: {median_loss:.7f}")
        print(f"   • Loss (reported) - Mean: {mean_output_loss:.7f}, Median: {median_output_loss:.7f}")
        print(f"   • Average loss: {avg_loss:.7f}")
        print(f"   • Accuracy: {accuracy:.5f} ({100*accuracy:.1f}%)")
        print(f"   • MSE: {avg_mse:.7f}")
        print(f"   • Precision: {precision:.5f}, Recall: {recall:.5f}, F1: {f1:.5f}")
        print(f"   • Time: {minutes} min {seconds:.3f} sec")
        
        return (avg_loss, accuracy, batch_losses, batch_indices, mse_batch_values,
                mean_loss, median_loss, mean_output_loss, median_output_loss,
                avg_mse, precision, recall, f1)
    
    def train(
        self,
        train_images: npt.NDArray[np.float32],
        train_labels: npt.NDArray[np.int64],
        test_images: Optional[npt.NDArray[np.float32]] = None,
        test_labels: Optional[npt.NDArray[np.int64]] = None,
        epochs: int = MY_CONSTANTS.epochs,
        batch_size: int = MY_CONSTANTS.batch_size,
        report_interval: int = 100,
        plot_batch_graphs: bool = False,
        device: torch.device = torch.device('cuda')
    ) -> Dict[str, List[float]]:
        """
        ПОЛНОЕ обучение сети на нескольких эпохах с сохранением всех метрик.
        
        Args:
            train_images: Тренировочные изображения
            train_labels: Тренировочные метки
            test_images: Тестовые изображения (опционально)
            test_labels: Тестовые метки (опционально)
            epochs: Количество эпох
            batch_size: Размер батча
            report_interval: Частота вывода отчета
            plot_batch_graphs: Рисовать ли графики по батчам
        
        Returns:
            Dict со всеми метриками по эпохам
        """
        self.train_losses = []
        self.test_losses = []
        self.train_accuracies = []
        self.test_accuracies = []
        self.train_mses = []
        self.test_mses = []
        self.train_precisions = []
        self.test_precisions = []
        self.train_recalls = []
        self.test_recalls = []
        self.train_f1s = []
        self.test_f1s = []
        self.epochs_completed = []
        print(f" STARTING TRAINING FOR {epochs} EPOCHS")
        
        for epoch in range(1, epochs + 1):
            print(f"EPOCH {epoch}/{epochs}")
            
            # Обучение на эпохе
            (avg_loss, accuracy, batch_losses, batch_indices, mse_values,
            mean_loss, median_loss, mean_out_loss, median_out_loss,
            avg_mse, precision, recall, f1) = self.train_epoch(
                train_images, train_labels, batch_size, report_interval, epoch, device
            )
            
            # СОХРАНЯЕМ МЕТРИКИ ТРЕНИРОВКИ
            self.train_losses.append(avg_loss)
            self.train_accuracies.append(accuracy)
            self.train_mses.append(avg_mse)
            self.train_precisions.append(precision)
            self.train_recalls.append(recall)
            self.train_f1s.append(f1)
            self.epochs_completed.append(epoch)
            
            # Сохраняем для графика по батчам
            self.batch_losses = batch_losses
            self.batch_indices = batch_indices
            
            print(f"✓ Training - Average loss: {avg_loss:.15f}, Accuracy: {accuracy:.15f} ({100*accuracy:.15f}%)")
            
            # Тестирование после эпохи
            if test_images is not None and test_labels is not None:
                test_accuracy, test_loss, test_batch_losses, test_avg_mse, test_precision, test_recall, test_f1 = self.predict(
                    test_images, test_labels, batch_size, verbose=False, device=device
                )
                self.test_losses.append(test_loss)
                self.test_accuracies.append(test_accuracy)
                self.test_mses.append(test_avg_mse)
                self.test_precisions.append(test_precision)
                self.test_recalls.append(test_recall)
                self.test_f1s.append(test_f1)
                
                print(f"\n📊 Test Results - Epoch {epoch}:")
                print(f"   • Loss: {test_loss:.7f}")
                print(f"   • Accuracy: {test_accuracy:.5f} ({100*test_accuracy:.1f}%)")
                print(f"   • MSE: {test_avg_mse:.7f}")
                print(f"   • Precision: {test_precision:.5f}")
                print(f"   • Recall: {test_recall:.5f}")
                print(f"   • F1: {test_f1:.5f}")
                print(f"✓ Testing  - Average loss: {test_loss:.15f}, Accuracy: {test_accuracy:.15f} ({100*test_accuracy:.15f}%)")
            
            # Опционально рисуем график по батчам
            if plot_batch_graphs:
                print_cycle_graph(batch_indices, batch_losses, epoch)
        
        print(f"\n✅ Training completed for {epochs} epochs!")
        
        return {
            'train_losses': self.train_losses,
            'train_accuracies': self.train_accuracies,
            'train_mses': self.train_mses,
            'train_precisions': self.train_precisions,
            'train_recalls': self.train_recalls,
            'train_f1s': self.train_f1s,
            'test_losses': self.test_losses,
            'test_accuracies': self.test_accuracies,
            'test_mses': self.test_mses,
            'test_precisions': self.test_precisions,
            'test_recalls': self.test_recalls,
            'test_f1s': self.test_f1s,
            'epochs': self.epochs_completed
        }

    """
    [EN] The implemention of test (predict) function.
    [RU] Имплементация функции-теста для нейронной сети
    """
    def predict(
        self,
        test_images: npt.NDArray[np.float32],
        test_labels: npt.NDArray[np.int64],
        batch_size: int = MY_CONSTANTS.batch_size,
        verbose: bool = True,
        device: torch.device = torch.device('cuda')
    ) -> Tuple[float, float, List[float]]:
        """
        Предсказание и оценка на тестовых данных.
        
        Args:
            test_images: Тестовые изображения
            test_labels: Тестовые метки
            batch_size: Размер батча
            verbose: Выводить ли детальную информацию
        
        Returns:
            Tuple[accuracy, avg_loss, batch_losses]
        """
        precision_metric = Precision(task="multiclass", num_classes=10, average='macro').to(device)
        recall_metric = Recall(task="multiclass", num_classes=10, average='macro').to(device)
        f1_metric = F1Score(task="multiclass", num_classes=10, average='macro').to(device)

        num_samples: int = test_images.shape[0]
        num_batches: int = (num_samples + batch_size - 1) // batch_size
        
        total_loss: float = 0.0
        correct_predictions: int = 0
        batch_losses: List[float] = []
        all_predictions = []  # Список для всех предсказаний
        all_labels = []      # Список для всех истинных меток
        total_mse = 0.0      # Накопление MSE по эпохе
        mse_batch_values = []  # Список MSE для каждого батча

        for i in range(0, num_samples, batch_size):
            batch_X: npt.NDArray[np.float32] = test_images[i:i + batch_size]
            batch_y: npt.NDArray[np.int64] = test_labels[i:i + batch_size]
            
            # Конвертируем в float64
            batch_X_f64: npt.NDArray[np.float64] = batch_X.astype(np.float64)

            # One-hot encoding для MSE
            batch_y_onehot: npt.NDArray[np.float64] = label_encode(batch_y).astype(np.float64)
            
            # Прямой проход
            output, _ = self.forward(batch_X_f64)
            
            # Loss
            loss: float = self.cross_entropy_loss(output, batch_y)
            total_loss += loss
            batch_losses.append(loss)
            
            # Точность
            predictions: npt.NDArray[np.int64] = np.argmax(output, axis=1)
            correct_predictions += np.sum(predictions == batch_y)

            # Метрики для scikit-learn варианта
            all_predictions.extend(predictions.tolist())
            all_labels.extend(batch_y.tolist())

            # Метрики для torchmetrics варианта
            pred_tensor = torch.from_numpy(predictions).to(device)
            target_tensor = torch.from_numpy(batch_y).to(device)
            
            precision_metric.update(pred_tensor, target_tensor)
            recall_metric.update(pred_tensor, target_tensor)
            f1_metric.update(pred_tensor, target_tensor)

            # Расчёт MSE для батча
            mse: float = float(np.mean(np.square(output - batch_y_onehot)))
            total_mse += mse
            mse_batch_values.append(mse)
        
        avg_loss: float = total_loss / num_batches
        accuracy: float = correct_predictions / num_samples
        
        if verbose:
            print(f"\nTest Error:")
            print(f"Accuracy: {(100*accuracy):>0.1f}%, Avg loss: {avg_loss:>8f}\n")
        
        avg_mse: float = total_mse / num_batches
    
        # Precision, Recall, F1-Score
        precision = precision_metric.compute().item()
        recall = recall_metric.compute().item()
        f1 = f1_metric.compute().item()
        
        precision_metric.reset()
        recall_metric.reset()
        f1_metric.reset()
        # precision: float = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
        # recall: float = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
        # f1: float = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
        
        # Статистики по loss
        mean_loss: float = float(np.mean(batch_losses))
        median_loss: float = float(np.median(batch_losses))
        
        if verbose:
            print(f"\n🔬 TESTING RESULTS:")
            print(f"   • Loss - Mean: {mean_loss:.7f}, Median: {median_loss:.7f}")
            print(f"   • Average loss: {avg_loss:.7f}")
            print(f"   • Accuracy: {(100*accuracy):>0.1f}%")
            print(f"   • MSE: {avg_mse:.7f}")
            print(f"   • Precision: {precision:.5f}")
            print(f"   • Recall: {recall:.5f}")
            print(f"   • F1-Score: {f1:.5f}")
        
        return accuracy, avg_loss, batch_losses, avg_mse, precision, recall, f1

    """
    [EN] Saves the model weights to a file using pickle library.
    [RU] Сохранение весов модели в файл с использованием библиотеки pickle
    """
    def save_model(
        self,
        filename_pickle: str = "../models/numpy_model.pkl",
        filename_joblib: str = "../models/numpy_model.joblib",
        filename_torch: str = "../models/numpy_model.pth"
    ) -> None:
        """Сохранение модели в файл."""
        weights: Dict[str, npt.NDArray[np.float64]] = {}
        for i, layer in enumerate(self.linear_layers):
            weights[f"layer_{i}_weights"] = layer.weights
            weights[f"layer_{i}_bias"] = layer.bias
        with open(filename_pickle, 'wb') as f:
            pickle.dump(weights, f)
        joblib.dump(weights, filename_joblib)
        torch.save(weights, torch_model_file)
        print(f"Saved Pickle NumPy Model State to {filename_pickle}")
        print(f"Saved Joblib NumPy Model State to {filename_joblib}")
        print(f"Saved Torch NumPy Model State to {filename_torch}")

    """
    [EN] Loads the model weights from a file.
    [RU] Загрузка весов модели из файла
    """
    def load_model(
        self,
        filename_pickle: str = "../models/numpy_model.pkl",
        filename_joblib: str = "../models/numpy_model.joblib",
        filename_torch: str = "../models/numpy_model.pth"
    ) -> None:
        """Загрузка модели из файла."""
        with open(filename_pickle, 'rb') as f:
            weights: Dict[str, npt.NDArray[np.float64]] = pickle.load(f)
        weights_joblib: Dict[str, npt.NDArray[np.float64]] = joblib.load(filename_joblib)
        weights_torch: Dict[str, npt.NDArray[np.float64]] = torch.load(filename_torch)
        for i, layer in enumerate(self.linear_layers):
            layer.weights = weights[f"layer_{i}_weights"]
            layer.bias = weights[f"layer_{i}_bias"]
        print(f"Loaded Pickle NumPy Model State from {filename_pickle}")
        print(f"Loaded Joblib NumPy Model State from {filename_joblib}")
        print(f"Loaded Torch NumPy Model State from {filename_torch}")

    """
    [EN] Function that help us to make a prediction by index
    [RU] Функция-предсказания для элемента по его индексу
    """
    def single_test(
        self,
        index: int,
        test_images: npt.NDArray[np.float32],
        test_labels: npt.NDArray[np.int64]
    ) -> None:
        """Тестирование на одном примере."""
        image: npt.NDArray[np.float64] = test_images[index].reshape(1, -1).astype(np.float64)
        label: int = test_labels[index]

        # Прямой ход
        output, _ = self.forward(image)
        predicted_label: str = labels_map[np.argmax(output)]
        print(f"Predicted: {predicted_label}, Actual: {labels_map[label]}")


### Уточнение - из чего складывается SGD (Stochastic Gradient Descent):
1. apply_gradients - Обновление весов с помощью SGD. КЛАССИЧЕСКАЯ ФОРМУЛА SGD: w = w - lr * grad, 
Формула: θ = θ - η * ∇θ J(θ)
где:
* θ - параметры (weights, bias)
* η - learning rate
* ∇θ J(θ) - градиент функции потерь
2. backward - Обратный проход (backpropagation) - вычисляет градиенты для SGD.
Градиент для Cross-Entropy + Softmax + усреднение.
3. Linear.backward - вычисление градиентов для линейного слоя.
Формулы:
- dL/dW = input.T @ grad_output
- dL/db = sum(grad_output, axis=0)
- dL/dx = grad_output @ weights.T

┌─────────────────────────────────────────────────────────┐
│                    SGD OPTIMIZER                        │
├─────────────────────────────────────────────────────────┤
│                                                         │
│  1. forward() ───► вычисление предсказаний              │
│         │                                               │
│         ▼                                               │
│  2. cross_entropy_loss() ───► вычисление ошибки         │
│         │                                               │
│         ▼                                               │
│  3. backward() ───► градиенты по цепному правилу        │
│         │                                               │
│         ├─────► Linear.backward() ───► dL/dW, dL/db     │
│         │                                               │
│         └─────► ReLU.backward()  ───► dL/dx             │
│                                                         │
│         ▼                                               │
│  4. apply_gradients() ───► w = w - lr * grad_w          │
│                            b = b - lr * grad_b          │
│                                                         │
└─────────────────────────────────────────────────────────┘